# Creating a Simple Agent with Tracing

In [5]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [1]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Nutrition Assistant Agent

In [8]:
nutrition_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful assistant giving out nutrition advice.
    You give concise answers.
    """,
)

placement_trainer = Agent(
    name="Placement Trainer",
    instructions="""
    You are a helpful placement trainer for college students.
    Guide them on resume building, aptitude practice, coding interview prep,
    communication, and mock interviews.
    Keep answers concise, practical, and encouraging.
    Give short step-by-step plans when asked for preparation advice.
    """,
)

Let's execute the Agent:

In [9]:
with trace("Simple Nutrition Agent"):
    result = await Runner.run(nutrition_agent, "How healthy are bananas?")

print(result)

RunResult:
- Last agent: Agent(name="Nutrition Assistant", ...)
- Final output (str):
    Bananas are generally healthy. They’re a good source of potassium, vitamin B6, vitamin C, and fiber, with little fat. They provide quick energy from natural sugars.
    
    Tips:
    - One medium banana (~105 calories) fits well in most diets; 1–2 daily is common.
    - Ripe bananas have more sugar but still a healthy choice; unripe have more resistant starch.
    - People with kidney disease or on potassium-restricted diets should monitor intake.
    - They are easy to combine with other foods (yogurt, oats, peanut butter) for balanced meals.
    
    Overall: a nutritious, convenient fruit when eaten as part of a varied diet.
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


In [10]:
with trace("Placement Trainer Agent"):
    result = await Runner.run(
        placement_trainer,
        "I have 2 weeks left for placement. What should I do every day?",
    )

print(result.final_output)

Great! Here’s a practical 14-day plan you can follow every day. It’s balanced between resume/CTC prep, coding/aptitude, interview practice, and mock interviews.

Daily template (adjust times to your schedule)
- 60–90 min: Aptitude + reasoning practice (quant, ver: 15–20 mins, data suff 15–20)
- 60–90 min: Coding practice (2 problems + review)
- 30–45 min: Resume/portfolio polish + targeted company research
- 30 min: Mock interview or speaking practice
- 15–30 min: Current affairs / HR questions + soft skills
- 7–8 hours sleep, 2x short breaks during study blocks

Day-by-day plan (two weeks)

Day 1
- Update resume: quantify impact (results, tech stack, projects)
- List target companies + typical interview rounds
- Practice 20 minutes of quant/logic (easy-medium)
- Solve 1 medium coding problem; read solution

Day 2
- Finish a concise 1-page summary of projects
- 1 coding problem (focus on arrays/strings)
- 15 minutes of HR/behavior questions (STAR)
- Do 1 mock experience question aloud


In [11]:
nutrition_agent = Agent(
    name="Student Assistant",
    instructions="""
    You are a helpful assistant giving out nutrition advice.
    You give concise answers.
    
    """,
)

Streaming the answer to the screen, token by token

In [4]:
response_stream = Runner.run_streamed(nutrition_agent, "How healthy are bananas?")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Generally healthy as part of a balanced diet.

Key points:
- Nutrients: good source of potassium, vitamin C, vitamin B6, and fiber; low in fat and protein.
- Benefits: supports heart health, digestion, and steady energy.
- Considerations: contains natural sugars; portion matters for those watching carbs or blood sugar.

Tips:
- A medium banana (~105 kcal) is fine daily for most people.
- Pair with protein/fat (e.g., peanut butter) to curb sugar spikes.
- Choose ripeness by preference: greener = less sugar, riper = sweeter.

Not a complete protein; eat a variety of foods for balance.

_Good Job!_